In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS streaming_upsert_demo (
        id LONG,
        value LONG,
        ingested_at TIMESTAMP,
        updated_at TIMESTAMP
    )
    USING DELTA
""")

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col
import time


def upsert_to_delta(batch_df, batch_id):
    """
    This function runs on each micro-batch.
    It performs an upsert (MERGE) — critical for CDC pipelines.
    """
    print(f"Processing batch {batch_id}, rows: {batch_df.count()}")
    
    target = DeltaTable.forName(spark, "streaming_upsert_demo")
    
    (
        target.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.id = s.id"  # match condition
        )
        .whenMatchedUpdate(set={
            "value": "s.value",
            "updated_at": "current_timestamp()"
        })
        .whenNotMatchedInsert(values={
            "id": "s.id",
            "value": "s.value",
            "ingested_at": "current_timestamp()",
            "updated_at": "current_timestamp()"
        })
        .execute()
    )

# Stream source
source_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 3)
    .load()
    .withColumn("id", col("value") % 10)  # only 10 unique IDs → forces updates
)

merge_query = (
    source_df.writeStream
    .foreachBatch(upsert_to_delta)   # ← the key pattern
    .option("checkpointLocation", "/Volumes/workspace/default/streaming_demo/checkpoints/merge")
    .trigger(availableNow=True)
    .start()
)

merge_query.awaitTermination()

# Prove upserts worked — should have max 10 rows
spark.table("streaming_upsert_demo").show()

In [0]:
spark.sql("DESCRIBE HISTORY streaming_upsert_demo") \
     .select("version", "timestamp", "operation", "operationMetrics") \
     .show(truncate=False)